# 03 — Intervention Decision Matrix

The model pipeline produces two scores per frustrated session:
- `p_frustrated` — LSTM frustration probability
- `p_churn` — LightGBM churn probability given frustration

The **DecisionEngine** multiplies these into a `combined_score` and gates
intervention type by (a) combined score tier and (b) user LTV.

**Firing condition:**
```
combined_score = p_frustrated × p_churn  >  fire_threshold
combined_score × LTV_30d  >  intervention_cost          (expected-value guard)
```

This notebook shows the decision logic, the LTV-tier × frustration-band matrix,
and the ROI calculations that justify each intervention tier.


In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Working dir:", os.getcwd())


Working dir: C:\Users\USER\emotional-friction-detector


In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap

FIGURES = "notebooks/figures"
os.makedirs(FIGURES, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# Intervention colours
ITYPE_COLORS = {
    "no_intervention":        "#ADB5BD",
    "empathy_message":        "#74C0FC",
    "voucher_rm2_plus_empathy": "#51CF66",
    "cs_escalation":          "#FF6B6B",
    "holdback":               "#E9ECEF",
}
print("Imports OK")


Imports OK


## 1. Score the Frustrated Cohort

Load `sessions_scored.parquet` (already contains LSTM `p_frustrated` scores)
and run the LightGBM churn model to get `p_churn`.


In [3]:
from models.lgbm_churn_risk.model import ChurnRiskModel, FEATURE_COLS

scored = pd.read_parquet("data/processed/sessions_scored.parquet")
model  = ChurnRiskModel.load("models/lgbm_churn_risk/artefacts/churn_model.lgb")

frust = scored[scored["is_frustrated"]].copy()
for c in FEATURE_COLS:
    if c not in frust.columns:
        frust[c] = 0.0

frust["p_churn"]   = model.predict_proba(frust[FEATURE_COLS])
frust["combined"]  = frust["p_frustrated"] * frust["p_churn"]

print(f"Frustrated sessions : {len(frust):,}")
print(f"p_frustrated  mean  : {frust['p_frustrated'].mean():.4f}")
print(f"p_churn       mean  : {frust['p_churn'].mean():.4f}")
print(f"combined_score mean : {frust['combined'].mean():.4f}  (max: {frust['combined'].max():.4f})")


Frustrated sessions : 79,333
p_frustrated  mean  : 0.9750
p_churn       mean  : 0.3885
combined_score mean : 0.3794  (max: 0.6525)


## 2. Score Distribution & Threshold Calibration

Our LSTM produces near-binary p_frustrated scores (simulation artefact of clean
synthetic patterns — production AUC target is >0.88, not 0.9986).  Because
`combined_score = p_frustrated × p_churn` is dominated by p_frustrated ≈ 1.0 for
confirmed frustrated sessions, the combined score effectively equals `p_churn`.

The DecisionEngine fire threshold (default 0.60) assumes more intermediate
p_frustrated values typical of a real LSTM (0.65–0.90 range).
For this portfolio demonstration we use simulation-calibrated thresholds:

| Threshold | Default (production) | Simulation-calibrated |
|-----------|---------------------|-----------------------|
| fire      | 0.60                | 0.30                  |
| voucher   | 0.75                | 0.40                  |
| escalation| 0.90                | 0.50                  |

The intervention logic and LTV-guard remain identical — only the numeric cutpoints
are shifted to match the simulation's score distribution.  In production, thresholds
are set by calibrating precision/recall targets on historical holdback data.


In [4]:
from intervention.decision_engine import DecisionEngine, ScoringResult, InterventionType
import random

# Calibrated thresholds for simulation score range
engine = DecisionEngine(
    fire_threshold=0.30,
    voucher_threshold=0.40,
    escalation_threshold=0.50,
    ltv_dormant_threshold_myr=30.0,
    holdback_rate=0.20,
)

# Plot combined_score distribution with threshold lines
fig, ax = plt.subplots(figsize=(9, 4))
bins = np.linspace(0, 0.7, 60)
ax.hist(frust["combined"], bins=bins, color="#457B9D", alpha=0.7, edgecolor="white")

for thresh, label, color in [
    (0.30, "fire (0.30)",      "#F4A261"),
    (0.40, "voucher (0.40)",   "#51CF66"),
    (0.50, "escalation (0.50)","#FF6B6B"),
]:
    ax.axvline(thresh, color=color, linewidth=2, linestyle="--", label=label)

ax.set_xlabel("combined_score  = p_frustrated x p_churn")
ax.set_ylabel("Session count")
ax.set_title("Combined score distribution — frustrated sessions (n=79,333)",
             fontweight="bold")
ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(f"{FIGURES}/03_combined_score_dist.png", bbox_inches="tight")
plt.close()
print("Saved 03_combined_score_dist.png")


Saved 03_combined_score_dist.png


In [5]:
random.seed(42)
np.random.seed(42)

decision_rows = []
for _, row in frust.iterrows():
    sr = ScoringResult(
        session_id=row["session_id"],
        user_id=row["user_id"],
        p_frustrated=float(row["p_frustrated"]),
        p_churn_given_frustrated=float(row["p_churn"]),
        ltv_estimate_myr=float(row["ltv_estimate_myr"]),
    )
    d = engine.decide(sr)
    decision_rows.append({
        "session_id":        row["session_id"],
        "user_id":           row["user_id"],
        "ltv_estimate_myr":  row["ltv_estimate_myr"],
        "p_frustrated":      row["p_frustrated"],
        "p_churn":           row["p_churn"],
        "combined":          d.combined_score,
        "intervention_type": d.intervention_type.value,
        "is_holdback":       d.is_holdback,
        "is_frustrated":     True,
    })

dec_df = pd.DataFrame(decision_rows)

print("Intervention distribution:")
itype_counts = dec_df["intervention_type"].value_counts()
for k, v in itype_counts.items():
    print(f"  {k:35s}: {v:6,}  ({v/len(dec_df)*100:.1f}%)")


Intervention distribution:
  voucher_rm2_plus_empathy           : 25,154  (31.7%)
  empathy_message                    : 19,441  (24.5%)
  no_intervention                    : 17,580  (22.2%)
  holdback                           : 12,383  (15.6%)
  cs_escalation                      :  4,775  (6.0%)


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Pie chart of intervention distribution
ax1 = axes[0]
labels_pie  = []
sizes_pie   = []
colors_pie  = []
order = ["no_intervention", "empathy_message", "voucher_rm2_plus_empathy",
         "cs_escalation", "holdback"]
for k in order:
    if k in itype_counts:
        labels_pie.append(k.replace("_", " ").title().replace("Rm2 ", "RM2 "))
        sizes_pie.append(itype_counts[k])
        colors_pie.append(ITYPE_COLORS[k])

wedges, texts, autotexts = ax1.pie(
    sizes_pie, labels=labels_pie, colors=colors_pie,
    autopct="%1.1f%%", startangle=140,
    wedgeprops=dict(edgecolor="white", linewidth=1.5),
    textprops=dict(fontsize=9),
)
for at in autotexts:
    at.set_fontsize(8)
ax1.set_title("Intervention distribution\n(79,333 frustrated sessions)", fontweight="bold")

# Bar chart: avg LTV by intervention type
ax2 = axes[1]
ltv_by_itype = dec_df.groupby("intervention_type")["ltv_estimate_myr"].mean().reindex(order).dropna()
colors_bar = [ITYPE_COLORS[k] for k in ltv_by_itype.index]
bars = ax2.bar(
    [k.replace("_", "\n").replace("voucher\nrm2\nplus\nempathy", "Voucher\n+Empathy")
     .replace("no\nintervention", "No\nIntervention")
     .replace("empathy\nmessage", "Empathy")
     .replace("cs\nescalation", "CS\nEscalation")
     .replace("holdback", "Holdback")
     for k in ltv_by_itype.index],
    ltv_by_itype.values, color=colors_bar, edgecolor="white"
)
ax2.axhline(30, color="grey", linestyle=":", linewidth=1.2, label="Dormant threshold (RM30)")
ax2.set_ylabel("Mean LTV (MYR)")
ax2.set_title("Mean user LTV by intervention tier\n(LTV-gating working correctly)",
              fontweight="bold")
ax2.legend(fontsize=9)
for bar, v in zip(bars, ltv_by_itype.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f"RM{v:.0f}", ha="center", fontsize=9, fontweight="bold")

fig.tight_layout()
fig.savefig(f"{FIGURES}/03_intervention_distribution.png", bbox_inches="tight")
plt.close()
print("Saved 03_intervention_distribution.png")


Saved 03_intervention_distribution.png


## 3. LTV-Tier × Frustration-Band Decision Matrix

The matrix shows the **dominant intervention type** in each cell, plus the
average churn risk (p_churn) to motivate why each tier fires what it does.

**Dormant guard:** Users with LTV < RM30 never receive vouchers — the at-risk
revenue is too low to justify the RM2 cost.  CS escalation is still permitted
(cost is ops time, not direct spend) when scores are very high.


In [7]:
# LTV tiers
ltv_bins   = [0, 30, 80, 150, 9999]
ltv_labels = ["Dormant\n(<RM30)", "Low\n(RM30-80)", "Med\n(RM80-150)", "High\n(>RM150)"]
frust_copy = dec_df.copy()
frust_copy["ltv_tier"] = pd.cut(frust_copy["ltv_estimate_myr"],
                                bins=ltv_bins, labels=ltv_labels)

# combined_score bands aligned to calibrated thresholds
combined_bins   = [0, 0.30, 0.40, 1.01]
combined_labels = ["Low (<0.30)\nNo action", "Med (0.30-0.40)\nEmpathy", "High (>0.40)\nVoucher/CS"]
frust_copy["score_band"] = pd.cut(frust_copy["combined"],
                                   bins=combined_bins, labels=combined_labels)

# Dominant intervention per cell
def dominant(s):
    vc = s.value_counts()
    return vc.index[0] if len(vc) else "no_intervention"

pivot_itype = frust_copy.groupby(
    ["ltv_tier", "score_band"], observed=True
)["intervention_type"].agg(dominant).unstack()

pivot_pchurn = frust_copy.groupby(
    ["ltv_tier", "score_band"], observed=True
)["p_churn"].mean().unstack()

pivot_count = frust_copy.groupby(
    ["ltv_tier", "score_band"], observed=True
).size().unstack()

print("Dominant intervention per cell:")
print(pivot_itype.to_string())
print()
print("Mean p_churn per cell:")
print(pivot_pchurn.round(3).to_string())
print()
print("Session count per cell:")
print(pivot_count.to_string())


Dominant intervention per cell:
score_band       Low (<0.30)\nNo action Med (0.30-0.40)\nEmpathy  High (>0.40)\nVoucher/CS
ltv_tier                                                                                  
Dormant\n(<RM30)        no_intervention          empathy_message           empathy_message
Low\n(RM30-80)          no_intervention          empathy_message  voucher_rm2_plus_empathy
Med\n(RM80-150)         no_intervention          empathy_message  voucher_rm2_plus_empathy
High\n(>RM150)          no_intervention          empathy_message  voucher_rm2_plus_empathy

Mean p_churn per cell:
score_band        Low (<0.30)\nNo action  Med (0.30-0.40)\nEmpathy  High (>0.40)\nVoucher/CS
ltv_tier                                                                                    
Dormant\n(<RM30)                   0.275                     0.365                     0.472
Low\n(RM30-80)                     0.273                     0.367                     0.476
Med\n(RM80-150)           

In [8]:
# Build the visual matrix
fig, ax = plt.subplots(figsize=(11, 5))
ax.set_xlim(0, len(combined_labels))
ax.set_ylim(0, len(ltv_labels))
ax.set_xticks(np.arange(len(combined_labels)) + 0.5)
ax.set_yticks(np.arange(len(ltv_labels)) + 0.5)
ax.set_xticklabels(combined_labels, fontsize=9)
ax.set_yticklabels(ltv_labels[::-1], fontsize=9)
ax.set_xlabel("Combined score band  (p_frustrated x p_churn)", fontsize=10)
ax.set_ylabel("LTV tier", fontsize=10)
ax.set_title("LTV-gated Intervention Decision Matrix", fontweight="bold", fontsize=12)
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)

# Intervention type -> short label + cost
CELL_LABELS = {
    "no_intervention":         ("No Action", "Cost: RM0", "ROI: —"),
    "empathy_message":         ("Empathy Msg", "Cost: RM0", "ROI: +NPS"),
    "voucher_rm2_plus_empathy":("Voucher RM2", "+ Empathy", "ROI: ~9x"),
    "cs_escalation":           ("CS Escalation", "Cost: ops", "ROI: +ret"),
    "holdback":                ("Holdback", "(A/B ctrl)", ""),
}

ltv_order  = ltv_labels[::-1]
for row_i, ltv in enumerate(ltv_order):
    for col_j, sband in enumerate(combined_labels):
        try:
            itype = pivot_itype.loc[ltv, sband] if (ltv in pivot_itype.index and sband in pivot_itype.columns) else "no_intervention"
            pchurn = pivot_pchurn.loc[ltv, sband] if (ltv in pivot_pchurn.index and sband in pivot_pchurn.columns) else 0.0
            count  = pivot_count.loc[ltv, sband]  if (ltv in pivot_count.index  and sband in pivot_count.columns) else 0
        except Exception:
            itype, pchurn, count = "no_intervention", 0.0, 0

        color = ITYPE_COLORS.get(str(itype), "#ADB5BD")
        rect  = plt.Rectangle([col_j, row_i], 1, 1, facecolor=color, edgecolor="white", linewidth=2)
        ax.add_patch(rect)

        lines = CELL_LABELS.get(str(itype), (str(itype), "", ""))
        ax.text(col_j + 0.5, row_i + 0.72, lines[0],
                ha="center", va="center", fontsize=9, fontweight="bold")
        ax.text(col_j + 0.5, row_i + 0.50, lines[1],
                ha="center", va="center", fontsize=8)
        ax.text(col_j + 0.5, row_i + 0.30, lines[2],
                ha="center", va="center", fontsize=8, color="#495057")
        ax.text(col_j + 0.5, row_i + 0.12, f"p_churn={pchurn:.2f}  n={count:,}",
                ha="center", va="center", fontsize=7, color="#495057")

# Legend
legend_patches = [
    mpatches.Patch(color=ITYPE_COLORS["no_intervention"],          label="No action"),
    mpatches.Patch(color=ITYPE_COLORS["empathy_message"],          label="Empathy message (RM0)"),
    mpatches.Patch(color=ITYPE_COLORS["voucher_rm2_plus_empathy"], label="Voucher RM2 + empathy"),
    mpatches.Patch(color=ITYPE_COLORS["cs_escalation"],            label="CS escalation"),
    mpatches.Patch(color=ITYPE_COLORS["holdback"],                 label="Holdback (A/B control)"),
]
ax.legend(handles=legend_patches, loc="upper right", bbox_to_anchor=(1.0, -0.14),
          ncol=5, fontsize=8, frameon=False)

fig.tight_layout()
fig.savefig(f"{FIGURES}/03_intervention_matrix.png", bbox_inches="tight")
plt.close()
print("Saved 03_intervention_matrix.png")


Saved 03_intervention_matrix.png


## 4. Expected Value Formula

The decision engine fires when:

$$EV = P(\text{frustrated}) \times P(\text{churn} | \text{frustrated}) \times \text{LTV}_{30d} - \text{cost} > 0$$

Equivalently: `combined_score × LTV_30d > cost`

With the intervention effects from the A/B simulator:
- **Empathy** reduces churn by 11% relative → RM0 cost → always fires when score qualifies
- **Voucher RM2** reduces churn by 23% relative → fires when saved LTV > RM2
- **CS escalation** reduces churn by 18% relative → fires for highest combined scores

**Dormant user guard:** LTV < RM30 → voucher EV is negative (RM2 cost vs RM30×0.40×0.23=RM2.76 save = marginal).
Voucher is downgraded to empathy to preserve ops budget.


In [9]:
# EV table by LTV tier
ltv_by_tier = {
    "Dormant (<RM30)":  18.4,
    "Low (RM30-80)":    56.2,
    "Med (RM80-150)":  113.8,
    "High (>RM150)":   278.7,
}
churn_reduction_voucher = 0.23
voucher_cost = 2.0

ev_rows = []
for tier, mean_ltv30d in ltv_by_tier.items():
    # Weighted mean p_churn for this tier across all score bands
    tier_data = frust_copy[frust_copy["ltv_tier"].astype(str).str.contains(tier.split("(")[0].strip())]
    p_churn_tier = tier_data["p_churn"].mean() if len(tier_data) > 0 else 0.40

    saved_ltv = p_churn_tier * mean_ltv30d * churn_reduction_voucher
    ev_voucher = saved_ltv - voucher_cost
    fires = "YES" if ev_voucher > 0 else "NO (dormant guard)"

    ev_rows.append({
        "LTV tier":         tier,
        "Mean LTV_30d (RM)":round(mean_ltv30d, 1),
        "Mean p_churn":     round(p_churn_tier, 3),
        "Saved LTV (RM)":   round(saved_ltv, 2),
        "Voucher EV (RM)":  round(ev_voucher, 2),
        "Fire voucher?":    fires,
    })

ev_df = pd.DataFrame(ev_rows)
print(ev_df.to_string(index=False))
print()
print(f"ROI (high-LTV user): {(ltv_by_tier['High (>RM150)'] * 0.40 * 0.23) / voucher_cost:.1f}x")
print(f"ROI (med-LTV user) : {(ltv_by_tier['Med (RM80-150)'] * 0.40 * 0.23) / voucher_cost:.1f}x")
print(f"ROI (low-LTV user) : {(ltv_by_tier['Low (RM30-80)'] * 0.40 * 0.23) / voucher_cost:.1f}x")

# EV bar chart
fig, ax = plt.subplots(figsize=(8, 3.5))
colors_ev = ["#FF6B6B" if ev < 0 else "#51CF66" for ev in ev_df["Voucher EV (RM)"]]
bars = ax.barh(ev_df["LTV tier"], ev_df["Voucher EV (RM)"], color=colors_ev, edgecolor="white")
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Expected value of RM2 voucher (RM)")
ax.set_title("Voucher EV by LTV tier\nPositive = fire; Negative = suppress (dormant guard)",
             fontweight="bold")
for bar, v in zip(bars, ev_df["Voucher EV (RM)"]):
    ax.text(v + 0.1 if v >= 0 else v - 0.1, bar.get_y() + bar.get_height()/2,
            f"RM{v:.2f}", va="center",
            ha="left" if v >= 0 else "right", fontsize=9)

fig.tight_layout()
fig.savefig(f"{FIGURES}/03_voucher_ev_by_tier.png", bbox_inches="tight")
plt.close()
print("\nSaved 03_voucher_ev_by_tier.png")


       LTV tier  Mean LTV_30d (RM)  Mean p_churn  Saved LTV (RM)  Voucher EV (RM)      Fire voucher?
Dormant (<RM30)               18.4         0.422            1.79            -0.21 NO (dormant guard)
  Low (RM30-80)               56.2         0.420            5.43             3.43                YES
 Med (RM80-150)              113.8         0.416           10.89             8.89                YES
  High (>RM150)              278.7         0.356           22.83            20.83                YES

ROI (high-LTV user): 12.8x
ROI (med-LTV user) : 5.2x
ROI (low-LTV user) : 2.6x

Saved 03_voucher_ev_by_tier.png


In [10]:
print("=" * 60)
print("INTERVENTION MATRIX KEY NUMBERS")
print("=" * 60)

total = len(dec_df)
n_fire = len(dec_df[dec_df["intervention_type"] != "no_intervention"])
n_voucher = len(dec_df[dec_df["intervention_type"] == "voucher_rm2_plus_empathy"])
n_empathy = len(dec_df[dec_df["intervention_type"] == "empathy_message"])
n_cs      = len(dec_df[dec_df["intervention_type"] == "cs_escalation"])
n_holdback= len(dec_df[dec_df["is_holdback"]])
n_no_int  = len(dec_df[dec_df["intervention_type"] == "no_intervention"])

print(f"Total frustrated sessions   : {total:,}")
print(f"  No intervention           : {n_no_int:,}  ({n_no_int/total*100:.1f}%)  combined < 0.30")
print(f"  Empathy message           : {n_empathy:,}  ({n_empathy/total*100:.1f}%)  combined 0.30-0.40")
print(f"  Voucher RM2 + empathy     : {n_voucher:,}  ({n_voucher/total*100:.1f}%)  combined 0.40-0.50, non-dormant")
print(f"  CS escalation             : {n_cs:,}  ({n_cs/total*100:.1f}%)  combined > 0.50")
print(f"  Holdback (A/B control)    : {n_holdback:,}  ({n_holdback/total*100:.1f}%)  20% of qualifying")
print()
print(f"Voucher ROI by LTV tier:")
for _, row in ev_df.iterrows():
    roi_str = f"RM{row['Voucher EV (RM)']:.2f} EV  ({row['Fire voucher?']})"
    print(f"  {row['LTV tier']:22s}: {roi_str}")
print()
print("Dormant users (LTV<30): voucher suppressed; CS escalation still allowed")
print()
print("Figures saved to notebooks/figures/:")
for f in sorted(os.listdir("notebooks/figures")):
    if f.startswith("03_"):
        print(f"  {f}")


INTERVENTION MATRIX KEY NUMBERS
Total frustrated sessions   : 79,333
  No intervention           : 17,580  (22.2%)  combined < 0.30
  Empathy message           : 19,441  (24.5%)  combined 0.30-0.40
  Voucher RM2 + empathy     : 25,154  (31.7%)  combined 0.40-0.50, non-dormant
  CS escalation             : 4,775  (6.0%)  combined > 0.50
  Holdback (A/B control)    : 12,383  (15.6%)  20% of qualifying

Voucher ROI by LTV tier:
  Dormant (<RM30)       : RM-0.21 EV  (NO (dormant guard))
  Low (RM30-80)         : RM3.43 EV  (YES)
  Med (RM80-150)        : RM8.89 EV  (YES)
  High (>RM150)         : RM20.83 EV  (YES)

Dormant users (LTV<30): voucher suppressed; CS escalation still allowed

Figures saved to notebooks/figures/:
  03_combined_score_dist.png
  03_intervention_distribution.png
  03_intervention_matrix.png
  03_voucher_ev_by_tier.png
